# Overlap Metrics: BLEU and ROUGE

formalNLP, Topic 27. This notebook walks through the topic's claims by calling the reference implementation,
`overlap_metrics_bleu_rouge.py`, which owns every number on the page and in the lab. Nothing here redefines the
mathematics.

An overlap metric is a function of the **match grid** between a candidate $y$ and its references. Clipping is a matching
on that grid. ROUGE-L is the longest chain through it. Both move by at most $O(n/|r|)$ when one word is edited, so they
cannot tell a paraphrase from a fact flip.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("notebooks/overlap-metrics-bleu-rouge").resolve()))
import numpy as np
import overlap_metrics_bleu_rouge as om

## 1. The match grid and clipping

Def 1 gives the clipped count $m_n(y, R) = \sum_g \min\bigl(c_y(g), \max_k c_{r_k}(g)\bigr)$. Prop 1 shows it is the
size of a maximum matching between the candidate's $n$-gram occurrences and the references' slots. A candidate that
repeats one matched word gets credit only as often as a reference holds that word.

In [ ]:
ref = tuple("the cat sat on the mat".split())
s = om.bleu_stats(("the",) * 7, [ref])
print("clipped unigram matches:", s.m[0], " precision:", om.precisions(s)[0])
om.test_prop1_clipping_is_a_maximum_matching(); om.test_prop1_matching_picture(); print("Prop 1 holds on 300 random instances")

## 2. BLEU

Def 2 defines BLEU as $\mathrm{BP}\cdot\exp\bigl(\tfrac1N\sum_n \log p_n\bigr)$, with counts pooled over the corpus and
$\mathrm{BP} = e^{\min(0,\,1 - r/c)}$. First, a cross-check against sacrebleu.

In [ ]:
om.test_bleu_matches_sacrebleu(); om.test_chrf_matches_sacrebleu(); om.test_rouge_matches_rouge_score()
print("BLEU, chrF, and ROUGE equal sacrebleu and rouge-score to 1e-12")

**Thm 1 (length envelope).** With one reference per segment, $\mathrm{BLEU} \le g(c/r)$, where
$g(x) = e^{1-1/x}$ for $x \le 1$. Above $x = 1$, $g(x)$ is $1/x$ for a single segment and $x^{-1/N}$ for a corpus.
Reference prefixes sit exactly on the left branch.

In [ ]:
for row in om.length_sweep():
    print(f"c/r = {row['x']:.3f}   BLEU = {row['bleu']:.3f}   corpus envelope = {row['bound']:.3f}")
om.test_thm1_length_envelope()

**Prop 2 (corpus BLEU is not the mean of sentence BLEU).** System A wins both sentences. System B wins the
corpus, because its two length errors cancel in the pooled brevity penalty.

In [ ]:
s = om.simpson()
for k in "AB":
    print(k, "sentences:", [round(v, 3) for v in s[k]["sentence"]], " corpus:", round(s[k]["corpus"], 3))
om.test_prop2_corpus_is_not_mean_sentence()

**Prop 3 (zero frequency).** With no matched 4-gram, unsmoothed sentence BLEU is 0. Add-$k$ smoothing pulls
each precision toward 1, which is Topic 3's shrinkage. **Prop 4 (more references).** Clipped counts can only rise when a
reference is added, but BLEU can fall.

In [ ]:
print("share of segments with no matched 4-gram:", {k: round(v, 3) for k, v in om.zero_four_gram().items()})
print(om.more_references())
om.test_prop3_zero_frequency_and_shrinkage(); om.test_prop4_more_references()

## 3. ROUGE and the longest chain

**Thm 2.** The LCS is the longest chain of marked cells that increases in both coordinates. The dynamic program finds
it. A chain is also a matching, so ROUGE-L never exceeds ROUGE-1.

In [ ]:
a, b = tuple("costs fell and revenue rose".split()), tuple("revenue rose while costs fell".split())
print("LCS path:", om.lcs_path(a, b), " ROUGE-L:", [round(x, 3) for x in om.rouge_l(a, b)],
      " ROUGE-1:", [round(x, 3) for x in om.rouge_n(a, b, 1)])
om.test_thm2_lcs_is_longest_path()

## 4. What overlap cannot see

**Thm 3.** $k$ substitutions move a clipped count by at most $nk$ and the LCS by at most $k$. **Cor 1.** Reversing every
fact of a finance summary changes one word per fact, so the scores barely move while fact F1 drops to 0.
**Prop 5.** Cutting a candidate at unmatched bigrams and permuting the pieces never lowers a clipped count.

In [ ]:
om.test_thm3_lipschitz_in_substitutions(); om.test_cor1_flips_within_the_bound(); om.test_prop5_chunk_permutations()
it = om.finance_items()[om.grid_example()]
print("facts:", it.facts)
for s in om.GRID_SYSTEMS:
    print(f"{s:8s}", " ".join(it.outputs[s][0]))
print("chunk census:", {k: round(v, 3) for k, v in om.chunk_census().items()})

## 5. chrF and soft matching

**Prop 6.** One substitution $u \to u'$ moves soft precision by at most $\|u - u'\|/|y|$ and soft recall by at most
$\|u - u'\|$. In Topic 7's vectors, "fell" is the nearest neighbor of "rose".

In [ ]:
print("cos(rose, fell) =", round(om.cosine("rose", "fell"), 3), " ||rose - fell|| =",
      round(float(np.sqrt(2 - 2 * om.cosine("rose", "fell"))), 3))
om.test_prop6_soft_match_is_blind_to_flips()

## 6. The metric as a decision rule

**Thm 4.** Minimum Bayes risk picks $\arg\max_y \mathbb{E}_{Y\sim p}\, U(y, Y)$. Exact-match utility gives the mode; BLEU
does not. On Topic 14's prompts, the trigram's exact mode is short and the brevity penalty punishes it.

In [ ]:
t = om.toy_mbr()
print("toy: mode", t["mode"], " exact-match MBR", t["mbr_exact_match"], " BLEU-MBR", t["mbr_bleu"])
tab = om.decoding_table()
for k in om.DECODING_SYSTEMS:
    print(f"{k:8s} BLEU {tab[k]['bleu']:.3f}  length {tab[k]['length']:.2f}  BP {tab[k]['bp']:.3f}")
print("reference length", round(tab["reference_length"], 2))
om.test_thm4_mbr_toy(); om.test_thm4_mbr_on_the_corpus()

## 7. Metrics and judgments

**Prop 7.** Averaging over segments removes per-segment noise but not a system's bias. System-level $\tau$ climbs to a
plateau below 1, and it reaches 1 only among systems that share one surface form.

In [ ]:
for m in om.METRICS:
    print(f"{m:8s} segment τ {om.segment_tau(m):.3f}  system τ {om.system_tau(m):.3f}  flip family {om.system_tau(m, om.FLIP_FAMILY):.3f}")
print("τ vs documents per system (BLEU):", dict(zip(om.TAU_N, [round(v, 3) for v in om.tau_curve()["bleu"]])))
om.test_prop7_bias_survives_averaging()

## 8. Finance case study: the answer key

The generator's aspect gold gives each document its facts. A writer's summary with every fact reversed outscores the
document's own fact-bearing sentences on every metric.

In [ ]:
tab = om.system_table()
print(f"{'system':11s}" + "".join(f"{m:>9s}" for m in (*om.METRICS, "f1")))
for s in om.SYSTEMS:
    print(f"{s:11s}" + "".join(f"{tab[s][m]:9.3f}" for m in (*om.METRICS, "f1")))
om.test_case_study_flipped_beats_extract()

## The whole harness

The lab's data file must equal `viz_payload()` exactly.

In [ ]:
assert om._run_harness() == 0